In [1]:
!pip install ipywidgets sentence-transformers scikit-learn numpy ttkbootstrap wordcloud matplotlib

In [ ]:
import tkinter as tk
from tkinter import scrolledtext, ttk, filedialog, messagebox
import ttkbootstrap as ttk
from ttkbootstrap.constants import *
import ttkbootstrap.tooltip as tooltip
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from wordcloud import WordCloud
import os

class TextSummarizerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Advanced Text Summarizer - Professional Edition")
        self.root.geometry("1200x800")
        self.root.minsize(800, 600)
        
        # Apply modern, light theme
        style = ttk.Style("litera")  # Light, clean theme
        style.configure("TButton", font=("Helvetica", 12, "bold"), padding=10, borderwidth=0)
        style.configure("TLabel", font=("Helvetica", 12))
        style.configure("TNotebook.Tab", font=("Helvetica", 14, "bold"), padding=[15, 8])
        style.configure("Title.TLabel", font=("Helvetica", 26, "bold"), foreground="#007BFF")  # Blue title
        style.configure("Custom.TFrame", background="#F8F9FA")  # Light grey background
        style.configure("Status.TLabel", font=("Helvetica", 10), background="#E9ECEF", foreground="#212529")
        
        # Custom button colors
        style.configure("primary.TButton", background="#007BFF", foreground="#FFFFFF")
        style.map("primary.TButton", background=[("active", "#0056B3")])  # Darker blue on hover
        style.configure("secondary.TButton", background="#6C757D", foreground="#FFFFFF")
        style.map("secondary.TButton", background=[("active", "#5A6268")])
        style.configure("info.TButton", background="#17A2B8", foreground="#FFFFFF")
        style.map("info.TButton", background=[("active", "#138496")])
        style.configure("success.TButton", background="#28A745", foreground="#FFFFFF")
        style.map("success.TButton", background=[("active", "#218838")])
        
        # Main frame
        main_frame = ttk.Frame(root, padding=20, style="Custom.TFrame")
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Title
        title_label = ttk.Label(main_frame, text="Advanced Text Summarizer", style="Title.TLabel")
        title_label.pack(pady=(0, 20))
        
        # Notebook for tabs
        notebook = ttk.Notebook(main_frame)
        notebook.pack(fill=tk.BOTH, expand=True, pady=15)
        
        # Input/Summary tab
        text_frame = ttk.Frame(notebook, padding=15, style="Custom.TFrame")
        notebook.add(text_frame, text="Text Input & Summary")
        
        # Visualization tab
        viz_frame = ttk.Frame(notebook, padding=15, style="Custom.TFrame")
        notebook.add(viz_frame, text="Visualizations")
        
        # Split input and output into side-by-side layout
        text_content_frame = ttk.Frame(text_frame, style="Custom.TFrame")
        text_content_frame.pack(fill=tk.BOTH, expand=True)
        
        # Input frame (left side)
        input_frame = ttk.LabelFrame(text_content_frame, text="Input Text", padding=10, style="Custom.TFrame")
        input_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 15))
        
        self.input_text = scrolledtext.ScrolledText(
            input_frame, 
            wrap=tk.WORD, 
            height=20, 
            background="#FFFFFF", 
            foreground="#212529", 
            font=("Helvetica", 11), 
            borderwidth=1, 
            relief="solid"
        )
        self.input_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Output frame (right side)
        output_frame = ttk.LabelFrame(text_content_frame, text="Summary", padding=10, style="Custom.TFrame")
        output_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(15, 0))
        
        self.output_text = scrolledtext.ScrolledText(
            output_frame, 
            wrap=tk.WORD, 
            height=20, 
            background="#FFFFFF", 
            foreground="#212529", 
            font=("Helvetica", 11), 
            borderwidth=1, 
            relief="solid"
        )
        self.output_text.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        # Control frame (below text areas)
        control_frame = ttk.Frame(text_frame, style="Custom.TFrame")
        control_frame.pack(fill=tk.X, pady=15)
        
        # Number of sentences selector
        sentences_label = ttk.Label(control_frame, text="Number of Sentences in Summary:", font=("Helvetica", 12, "bold"), foreground="#212529")
        sentences_label.pack(side=tk.LEFT, padx=(0, 15))
        
        self.num_sentences = tk.StringVar(value="3")
        sentences_spinbox = ttk.Spinbox(control_frame, from_=1, to=20, width=5, textvariable=self.num_sentences, font=("Helvetica", 11))
        sentences_spinbox.pack(side=tk.LEFT, padx=15)
        tooltip.ToolTip(sentences_spinbox, text="Select the number of sentences for the summary (1-20).")
        
        # Buttons
        self.summarize_btn = ttk.Button(control_frame, text="Summarize", command=self.summarize_text, style="primary.TButton")
        self.summarize_btn.pack(side=tk.LEFT, padx=15)
        tooltip.ToolTip(self.summarize_btn, text="Generate a summary and visualizations for the input text.")
        
        self.clear_btn = ttk.Button(control_frame, text="Clear", command=self.clear_text, style="secondary.TButton")
        self.clear_btn.pack(side=tk.LEFT, padx=15)
        tooltip.ToolTip(self.clear_btn, text="Clear the input, summary, and visualizations.")
        
        self.open_btn = ttk.Button(control_frame, text="Open File", command=self.open_file, style="info.TButton")
        self.open_btn.pack(side=tk.LEFT, padx=15)
        tooltip.ToolTip(self.open_btn, text="Load a text file into the input area.")
        
        self.save_btn = ttk.Button(control_frame, text="Save Summary", command=self.save_summary, style="success.TButton")
        self.save_btn.pack(side=tk.LEFT, padx=15)
        tooltip.ToolTip(self.save_btn, text="Save the summary to a text file.")
        
        # Visualization canvas
        self.viz_canvas_frame = ttk.Frame(viz_frame, style="Custom.TFrame", borderwidth=1, relief="solid")
        self.viz_canvas_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Status bar
        self.status_var = tk.StringVar()
        self.status_bar = ttk.Label(main_frame, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W, style="Status.TLabel")
        self.status_bar.pack(fill=tk.X, side=tk.BOTTOM, pady=(10, 0))
        self.status_var.set("Ready")
        
        # Initialize sentence transformer model
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Placeholder for visualization canvas
        self.viz_canvas = None
    
    def summarize_text(self):
        """Summarize the text, generate visualizations, and display in output area."""
        text = self.input_text.get("1.0", tk.END).strip()
        
        if not text:
            self.status_var.set("Warning: Please enter some text to summarize!")
            self.status_bar.configure(foreground="#DC3545")  # Red for warnings
            messagebox.showwarning("Warning", "Please enter some text to summarize!")
            return
        
        try:
            num_sentences = int(self.num_sentences.get())
            if num_sentences < 1:
                num_sentences = 1
        except ValueError:
            num_sentences = 3
            self.num_sentences.set("3")
        
        summary = self.improved_summarize(text, num_sentences)
        
        self.output_text.delete("1.0", tk.END)
        self.output_text.insert(tk.END, summary)
        self.status_var.set(f"Summarized: {len(text.split())} words to {len(summary.split())} words")
        self.status_bar.configure(foreground="#28A745")  # Green for success
        
        # Generate and display visualizations
        self.generate_visualizations(text)
    
    def improved_summarize(self, text, num_sentences=3):
        """
        An advanced text summarizer using TF-IDF, sentence embeddings, and clustering.
        
        Args:
            text (str): The text to summarize
            num_sentences (int): Number of sentences to include in summary
            
        Returns:
            str: The summarized text
        """
        # Split text into sentences
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if s.strip() and len(s.split()) > 3]
        
        # If text is too short, return original
        if not sentences:
            self.status_var.set("No valid sentences found for summarization.")
            self.status_bar.configure(foreground="#DC3545")
            return text
        if len(sentences) <= num_sentences:
            self.status_var.set("Text is already concise. No summarization needed.")
            self.status_bar.configure(foreground="#28A745")
            return ' '.join(sentences)
        
        # Compute TF-IDF scores
        vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
        try:
            tfidf_matrix = vectorizer.fit_transform(sentences)
            tfidf_scores = np.sum(tfidf_matrix.toarray(), axis=1)
        except ValueError:
            tfidf_scores = np.ones(len(sentences))  # Fallback for empty vocabulary
        
        # Compute sentence embeddings
        embeddings = self.model.encode(sentences, convert_to_numpy=True)
        
        # Normalize TF-IDF scores
        max_tfidf = max(tfidf_scores) if max(tfidf_scores) > 0 else 1
        tfidf_scores = tfidf_scores / max_tfidf
        
        # Cluster sentences to ensure diversity
        num_clusters = min(num_sentences, len(sentences))
        kmeans = KMeans(n_clusters=num_clusters, random_state=42)
        cluster_labels = kmeans.fit_predict(embeddings)
        
        # Score sentences based on TF-IDF, position, and semantic centrality
        sentence_scores = {}
        for i, sentence in enumerate(sentences):
            score = tfidf_scores[i] * 0.5  # Weight TF-IDF
            
            # Position-based scoring
            if i == 0 or i == len(sentences) - 1:
                score += 0.2
            
            # Semantic centrality
            cluster_center = kmeans.cluster_centers_[cluster_labels[i]]
            semantic_score = 1 / (1 + np.linalg.norm(embeddings[i] - cluster_center))
            score += semantic_score * 0.3
            
            # Boost for importance markers
            importance_markers = [
                "important", "significant", "result", "conclude", "summary", 
                "therefore", "thus", "in conclusion", "finally", "key", "main", 
                "primary", "essential", "critical", "crucial", "major", "vital"
            ]
            for marker in importance_markers:
                if marker in sentence.lower():
                    score += 0.2
            
            sentence_scores[i] = score
        
        # Select top sentences from different clusters
        selected_indices = []
        cluster_counts = Counter(cluster_labels)
        for cluster in range(num_clusters):
            if len(selected_indices) >= num_sentences:
                break
            cluster_indices = [i for i in range(len(sentences)) if cluster_labels[i] == cluster]
            if not cluster_indices:
                continue
            cluster_indices = sorted(cluster_indices, key=lambda x: sentence_scores[x], reverse=True)
            selected_indices.append(cluster_indices[0])
        
        # Fill remaining slots
        remaining_indices = [i for i in range(len(sentences)) if i not in selected_indices]
        remaining_indices = sorted(remaining_indices, key=lambda x: sentence_scores[x], reverse=True)
        selected_indices.extend(remaining_indices[:num_sentences - len(selected_indices)])
        
        # Sort indices for coherence
        selected_indices = sorted(selected_indices)
        
        # Create summary
        summary = ' '.join(sentences[i] for i in selected_indices)
        return summary
    
    def generate_visualizations(self, text):
        """Generate and display bar chart and word cloud for the input text."""
        # Clear previous visualizations
        if self.viz_canvas:
            self.viz_canvas.get_tk_widget().destroy()
        
        # Extract words, filter stopwords, special characters, and numbers
        stopwords = set([
            "a", "an", "the", "and", "or", "but", "if", "because", "as", "what", "which", 
            "this", "that", "these", "those", "then", "just", "so", "than", "such", "both", 
            "through", "about", "for", "is", "of", "while", "during", "to", "from", "in", 
            "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", 
            "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", 
            "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", 
            "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", 
            "should", "now"
        ])
        # Match only alphabetic words, excluding numbers and special characters
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        word_freq = Counter([word for word in words if word not in stopwords and len(word) > 2])
        
        if not word_freq:
            self.status_var.set("Warning: No significant words found for visualization!")
            self.status_bar.configure(foreground="#DC3545")
            messagebox.showwarning("Warning", "No significant words found for visualization!")
            return
        
        # Prepare figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
        fig.suptitle("Text Analysis Visualizations", fontsize=16, fontweight="bold", color="#212529")
        fig.patch.set_facecolor("#F8F9FA")  # Light background for visualizations
        
        # Bar chart of top 10 words
        top_words = word_freq.most_common(10)
        words, counts = zip(*top_words)
        ax1.bar(words, counts, color="#007BFF", edgecolor="#FFFFFF")
        ax1.set_title("Top 10 Frequent Words", fontsize=14, fontweight="bold", color="#212529")
        ax1.set_xlabel("Words", fontsize=12, color="#212529")
        ax1.set_ylabel("Frequency", fontsize=12, color="#212529")
        ax1.set_facecolor("#FFFFFF")
        ax1.tick_params(axis="x", rotation=45, colors="#212529")
        ax1.tick_params(axis="y", colors="#212529")
        ax1.grid(True, axis="y", linestyle="--", alpha=0.3, color="#6C757D")
        
        # Word cloud
        wordcloud = WordCloud(
            width=600, height=400, background_color="#FFFFFF", 
            colormap="Blues", min_font_size=10, max_font_size=150, 
            font_path=None, stopwords=stopwords
        ).generate_from_frequencies(word_freq)
        ax2.imshow(wordcloud, interpolation="bilinear")
        ax2.set_title("Word Cloud", fontsize=14, fontweight="bold", color="#212529")
        ax2.axis("off")
        
        # Adjust layout
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        
        # Embed in Tkinter
        self.viz_canvas = FigureCanvasTkAgg(fig, master=self.viz_canvas_frame)
        self.viz_canvas.draw()
        self.viz_canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        
        # Save visualizations
        output_dir = "summarizer_visualizations"
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, "text_analysis.png")
        fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
        self.status_var.set(f"Visualizations saved to: {output_path}")
        self.status_bar.configure(foreground="#28A745")
        
        # Close figure to free memory
        plt.close(fig)
    
    def clear_text(self):
        """Clear both input and output text areas and visualizations."""
        self.input_text.delete("1.0", tk.END)
        self.output_text.delete("1.0", tk.END)
        if self.viz_canvas:
            self.viz_canvas.get_tk_widget().destroy()
            self.viz_canvas = None
        self.status_var.set("Ready")
        self.status_bar.configure(foreground="#212529")
    
    def open_file(self):
        """Open a text file and load it into the input area."""
        file_path = filedialog.askopenfilename(
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
        )
        
        if file_path:
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()
                
                self.input_text.delete("1.0", tk.END)
                self.input_text.insert(tk.END, content)
                self.status_var.set(f"Loaded file: {file_path}")
                self.status_bar.configure(foreground="#28A745")
            except Exception as e:
                self.status_var.set(f"Error: Could not open file: {e}")
                self.status_bar.configure(foreground="#DC3545")
                messagebox.showerror("Error", f"Could not open file: {e}")
    
    def save_summary(self):
        """Save the summary to a text file."""
        summary = self.output_text.get("1.0", tk.END).strip()
        
        if not summary:
            self.status_var.set("Warning: No summary to save!")
            self.status_bar.configure(foreground="#DC3545")
            messagebox.showwarning("Warning", "No summary to save!")
            return
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
        )
        
        if file_path:
            try:
                with open(file_path, 'w', encoding='utf-8') as file:
                    file.write(summary)
                self.status_var.set(f"Summary saved to: {file_path}")
                self.status_bar.configure(foreground="#28A745")
            except Exception as e:
                self.status_var.set(f"Error: Could not save file: {e}")
                self.status_bar.configure(foreground="#DC3545")
                messagebox.showerror("Error", f"Could not save file: {e}")

def main():
    root = ttk.Window(themename="litera")
    app = TextSummarizerApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()